In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps -e {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=(
        "assets/models/qwen3.5-4b/",
        "assets/lenses/qwen3.5-4b/",
    ),
)

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from experiments.jlens_readout_sanity.experiment import (
    Case,
    ExperimentRuntime,
    InterventionSpec,
    ReadoutSpec,
)

CASES = (
    Case(
        key="spider",
        prompt="The number of legs on the animal that spins webs is",
        expected_answers=("8", "eight"),
        readout=ReadoutSpec(
            concepts=("spider",),
            require_capability_gate=True,
        ),
        intervention=InterventionSpec(
            source_surface=" spider",
            target_surface=" ant",
            target_answers=("6", "six"),
        ),
    ),
    Case(
        key="france_capital",
        prompt="The capital of France is the city of",
        expected_answers=("Paris",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Beijing",)),
    ),
    Case(
        key="france_language",
        prompt="Most people in France speak",
        expected_answers=("French",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Chinese",)),
    ),
    Case(
        key="france_continent",
        prompt="France is a country on the continent of",
        expected_answers=("Europe",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Asia",)),
    ),
    Case(
        key="france_currency",
        prompt="The single-word name for the currency now used in France is the",
        expected_answers=("Euro",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Yuan",)),
    ),
)

In [ ]:
import importlib.metadata

import jlens
import torch
import transformers

from experiments.jlens_readout_sanity.constants import (
    LENS_PATH,
    MODEL_PATH,
)
from experiments.jlens_readout_sanity.utils import (
    render_sanity_report,
    run_experiment,
    validate_model_lens,
    write_results,
)
from jlens_reasoning.evaluation import GenerationStatus, ModelOutput

model_path = Path(MODEL_PATH)
lens_path = Path(LENS_PATH)
missing_assets = [path for path in (model_path, lens_path) if not path.exists()]
if missing_assets:
    missing = ", ".join(str(path) for path in missing_assets)
    raise FileNotFoundError(f"Missing downloaded assets: {missing}")

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    local_files_only=True,
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True,
)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(str(lens_path))
validate_model_lens(model, lens)
model, lens

In [ ]:
@torch.inference_mode()
def forward_next_token(input_ids):
    return causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]


@torch.inference_mode()
def generate_output(prompt: str) -> ModelOutput:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(context.device)
    generated = causal_lm.generate(
        input_ids=input_ids,
        do_sample=False,
        max_new_tokens=64,
    )
    generated_ids = generated[0, input_ids.shape[1] :].tolist()
    eos_ids = causal_lm.generation_config.eos_token_id
    eos_token_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or ())
    generation_status = (
        GenerationStatus.COMPLETE
        if generated_ids and generated_ids[-1] in eos_token_ids
        else GenerationStatus.TRUNCATED
    )
    text_ids = (
        generated_ids[:-1]
        if generation_status is GenerationStatus.COMPLETE
        else generated_ids
    )
    return ModelOutput(
        text=tokenizer.decode(text_ids, skip_special_tokens=True),
        token_ids=tuple(generated_ids),
        token_pieces=tuple(
            tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
            for token_id in generated_ids
        ),
        generation_status=generation_status,
        finish_reason=(
            "eos" if generation_status is GenerationStatus.COMPLETE else "length"
        ),
    )


runtime = ExperimentRuntime(
    model=model,
    lens=lens,
    tokenizer=tokenizer,
    unembedding_weight=causal_lm.get_output_embeddings().weight,
    forward_next_token=forward_next_token,
    generate_output=generate_output,
)
result = run_experiment(cases=CASES, runtime=runtime)

In [ ]:
result.provenance = {
    "source_sha256": PROJECT_SOURCE_SHA256,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "jlens": importlib.metadata.version("jlens"),
}

run_dir = OUTPUT_DIR / "runs/jlens-readout-sanity"
result_path = run_dir / "result.json"
write_results(result_path, result)
print(f"Saved: {result_path}")

In [ ]:
print(render_sanity_report(result))

In [ ]:
if not result.passed:
    raise RuntimeError(
        "Read-and-change sanity checks failed: " + "; ".join(result.failures)
    )

print("All J-Lens read-and-change sanity checks passed.")